In [2]:
# phase4_analysis.py
# Goal: Load our clean CSV and extract meaningful insights

import pandas as pd

# --- LOAD THE CSV ---
df = pd.read_csv("weather_data_20260414_1052.csv")
# replace filename with whatever yours is actually called

print("--- DATA LOADED ---")
print(f"Shape: {df.shape}")
print(df.head())

--- DATA LOADED ---
Shape: (15, 16)
        city country         collected_at  temp_c  feels_like_c  \
0    Lucknow      IN  2026-04-14 10:50:58   33.99         32.15   
1      Delhi      IN  2026-04-14 10:50:58   32.05         30.37   
2     Mumbai      IN  2026-04-14 10:50:58   33.99         39.06   
3  Bengaluru      IN  2026-04-14 10:50:58   32.09         33.18   
4    Kolkata      IN  2026-04-14 10:50:58   31.97         38.97   

   feels_like_diff temp_category  temp_min_c  temp_max_c  humidity  \
0             -1.8           Hot       33.99       33.99        22   
1             -1.7          Warm       32.05       32.05        25   
2              5.1           Hot       30.94       33.99        52   
3              1.1          Warm       31.12       33.06        44   
4              7.0          Warm       31.97       31.97        70   

  humidity_level  pressure_hpa  wind_speed_mps  visibility_m condition  \
0            Dry          1008            3.09          5000      

In [3]:
# --- EDA CHECKLIST ---

print("\n--- SHAPE ---")
print(df.shape)                    # how many rows and columns

print("\n--- COLUMN TYPES ---")
print(df.dtypes)                   # what type is each column

print("\n--- MISSING VALUES ---")
print(df.isnull().sum())           # any nulls remaining?

print("\n--- SUMMARY STATS ---")
print(df.describe().round(2))      # mean, min, max, std for all numbers

print("\n--- UNIQUE VALUES ---")
print(df["condition"].value_counts())   # what weather conditions exist?


--- SHAPE ---
(15, 16)

--- COLUMN TYPES ---
city                object
country             object
collected_at        object
temp_c             float64
feels_like_c       float64
feels_like_diff    float64
temp_category       object
temp_min_c         float64
temp_max_c         float64
humidity             int64
humidity_level      object
pressure_hpa         int64
wind_speed_mps     float64
visibility_m         int64
condition           object
description         object
dtype: object

--- MISSING VALUES ---
city               0
country            0
collected_at       0
temp_c             0
feels_like_c       0
feels_like_diff    0
temp_category      0
temp_min_c         0
temp_max_c         0
humidity           0
humidity_level     0
pressure_hpa       0
wind_speed_mps     0
visibility_m       0
condition          0
description        0
dtype: int64

--- SUMMARY STATS ---
       temp_c  feels_like_c  feels_like_diff  temp_min_c  temp_max_c  \
count   15.00         15.00            1

In [4]:
# --- Q1: TEMPERATURE RANKING ---
print("\n--- TEMPERATURE RANKING (Hottest → Coldest) ---")

temp_ranking = df[["city", "country", "temp_c", "temp_category"]]\
               .sort_values("temp_c", ascending=False)\
               .reset_index(drop=True)

temp_ranking.index += 1            # start ranking from 1 not 0
print(temp_ranking.to_string())

# Top 3 hottest
print(f"\nHottest city  : {df.loc[df['temp_c'].idxmax(), 'city']} "
      f"({df['temp_c'].max()}°C)")

# Top 3 coldest
print(f"Coldest city  : {df.loc[df['temp_c'].idxmin(), 'city']} "
      f"({df['temp_c'].min()}°C)")

# Average temperature
print(f"Average temp  : {df['temp_c'].mean().round(1)}°C")


--- TEMPERATURE RANKING (Hottest → Coldest) ---
         city country  temp_c temp_category
1        Pune      IN   37.30           Hot
2   Hyderabad      IN   35.23           Hot
3      Mumbai      IN   33.99           Hot
4     Lucknow      IN   33.99           Hot
5     Chennai      IN   33.25           Hot
6   Ahmedabad      IN   33.02           Hot
7      Jaipur      IN   32.62          Warm
8   Bengaluru      IN   32.09          Warm
9       Delhi      IN   32.05          Warm
10    Kolkata      IN   31.97          Warm
11      Dubai      AE   24.90          Mild
12      Tokyo      JP   23.58          Mild
13     Sydney      AU   23.19          Mild
14   New York      US   19.85          Mild
15     London      GB    3.70          Cold

Hottest city  : Pune (37.3°C)
Coldest city  : London (3.7°C)
Average temp  : 28.7°C


In [5]:
# --- Q2: INDIA VS WORLD COMPARISON ---
print("\n--- INDIA VS GLOBAL CITIES ---")

india_df  = df[df["country"] == "IN"]
global_df = df[df["country"] != "IN"]

comparison = pd.DataFrame({
    "metric"        : ["Avg Temp (°C)", "Avg Humidity (%)", 
                       "Avg Wind (m/s)", "City Count"],
    "India"         : [
        india_df["temp_c"].mean().round(1),
        india_df["humidity"].mean().round(1),
        india_df["wind_speed_mps"].mean().round(1),
        len(india_df)
    ],
    "Global"        : [
        global_df["temp_c"].mean().round(1),
        global_df["humidity"].mean().round(1),
        global_df["wind_speed_mps"].mean().round(1),
        len(global_df)
    ]
})

print(comparison.to_string(index=False))


--- INDIA VS GLOBAL CITIES ---
          metric  India  Global
   Avg Temp (°C)   33.6    19.0
Avg Humidity (%)   35.7    63.0
  Avg Wind (m/s)    2.6     3.7
      City Count   10.0     5.0


In [6]:
# --- Q3: GROUP BY WEATHER CONDITION ---
print("\n--- STATS BY WEATHER CONDITION ---")

condition_stats = df.groupby("condition").agg(
    city_count    = ("city",          "count"),
    avg_temp      = ("temp_c",        "mean"),
    avg_humidity  = ("humidity",      "mean"),
    avg_wind      = ("wind_speed_mps","mean"),
    cities        = ("city",          lambda x: ", ".join(x))
).round(1)

print(condition_stats.to_string())


--- STATS BY WEATHER CONDITION ---
           city_count  avg_temp  avg_humidity  avg_wind                                      cities
condition                                                                                          
Clear               5      23.2          51.4       2.3   Bengaluru, Pune, London, New York, Sydney
Clouds              3      27.2          57.3       5.3                       Chennai, Tokyo, Dubai
Haze                5      33.2          33.2       2.2  Lucknow, Delhi, Kolkata, Hyderabad, Jaipur
Smoke               2      33.5          38.5       3.1                           Mumbai, Ahmedabad


Question 4 — Does humidity affect how hot it feels?

In [7]:
# --- Q4: CORRELATION ANALYSIS ---
print("\n--- CORRELATION MATRIX ---")

# Select only numeric columns for correlation
numeric_cols = ["temp_c", "feels_like_c", "humidity", 
                "pressure_hpa", "wind_speed_mps", "feels_like_diff"]

corr_matrix = df[numeric_cols].corr().round(2)
print(corr_matrix)

# Pull out the most interesting correlations
print("\n--- KEY CORRELATIONS WITH TEMPERATURE ---")
temp_corr = corr_matrix["temp_c"].drop("temp_c").sort_values(ascending=False)
print(temp_corr)


--- CORRELATION MATRIX ---
                 temp_c  feels_like_c  humidity  pressure_hpa  wind_speed_mps  \
temp_c             1.00          0.95     -0.73         -0.69            0.01   
feels_like_c       0.95          1.00     -0.47         -0.68            0.06   
humidity          -0.73         -0.47      1.00          0.47            0.13   
pressure_hpa      -0.69         -0.68      0.47          1.00            0.26   
wind_speed_mps     0.01          0.06      0.13          0.26            1.00   
feels_like_diff    0.17          0.48      0.53         -0.21            0.16   

                 feels_like_diff  
temp_c                      0.17  
feels_like_c                0.48  
humidity                    0.53  
pressure_hpa               -0.21  
wind_speed_mps              0.16  
feels_like_diff             1.00  

--- KEY CORRELATIONS WITH TEMPERATURE ---
feels_like_c       0.95
feels_like_diff    0.17
wind_speed_mps     0.01
pressure_hpa      -0.69
humidity          -0

In [8]:
# --- Q5: CITY COMFORT INDEX ---
print("\n--- CITY COMFORT INDEX ---")

# Comfort logic:
# ideal temp = 22°C, ideal humidity = 50%
# the closer to ideal, the higher the score

df["temp_score"]     = 100 - abs(df["temp_c"] - 22) * 2
df["humidity_score"] = 100 - abs(df["humidity"] - 50) * 0.5
df["comfort_index"]  = ((df["temp_score"] + df["humidity_score"]) / 2).round(1)

comfort_ranking = df[["city", "country", "temp_c", "humidity", "comfort_index"]]\
                  .sort_values("comfort_index", ascending=False)\
                  .reset_index(drop=True)
comfort_ranking.index += 1

print(comfort_ranking.to_string())
print(f"\nMost comfortable city : {comfort_ranking.iloc[0]['city']}")
print(f"Least comfortable city: {comfort_ranking.iloc[-1]['city']}")


--- CITY COMFORT INDEX ---
         city country  temp_c  humidity  comfort_index
1      Sydney      AU   23.19        50           98.8
2       Tokyo      JP   23.58        49           98.2
3    New York      US   19.85        65           94.1
4       Dubai      AE   24.90        63           93.8
5   Bengaluru      IN   32.09        44           88.4
6      Mumbai      IN   33.99        52           87.5
7     Chennai      IN   33.25        60           86.2
8     Kolkata      IN   31.97        70           85.0
9       Delhi      IN   32.05        25           83.7
10  Ahmedabad      IN   33.02        25           82.7
11  Hyderabad      IN   35.23        31           82.0
12     Jaipur      IN   32.62        18           81.4
13    Lucknow      IN   33.99        22           81.0
14       Pune      IN   37.30        10           74.7
15     London      GB    3.70        88           72.2

Most comfortable city : Sydney
Least comfortable city: London


In [9]:
# --- SAVE ANALYSIS RESULTS ---

# Save the comfort ranking
comfort_ranking.to_csv("city_comfort_ranking.csv", index=False)

# Save condition summary
condition_stats.to_csv("condition_summary.csv")

# Save the full analyzed dataframe
df.to_csv("weather_analyzed.csv", index=False)

print("\nAll analysis files saved!")
print("  → weather_analyzed.csv")
print("  → city_comfort_ranking.csv")
print("  → condition_summary.csv")


All analysis files saved!
  → weather_analyzed.csv
  → city_comfort_ranking.csv
  → condition_summary.csv
